In [46]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

spark = SparkSession.builder \
    .appName("KMeansCafe") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/28 09:44:57 WARN Utils: Your hostname, Laptop-de-Gabriel.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.101 instead (on interface en0)
26/05/28 09:44:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/28 09:44:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [47]:
import glob
import pandas as pd
from functools import reduce
import os
import geopandas as gpd

pasta_arabica = 'dados-brutos/arabica'
pasta_canephora = 'dados-brutos/canephora'

arquivos_arabica = glob.glob(os.path.join(pasta_arabica, '*.xlsx'))
arquivos_canephora = glob.glob(os.path.join(pasta_canephora, '*.xlsx'))

In [48]:
def trata_excel(caminho):
    
    df = pd.read_excel(caminho, header=None)
    
    linha1 = df.iloc[1]  # variável
    linha3 = df.iloc[3]  # ano
    linha4 = df.iloc[4]  # tipo
    
    linha1 = linha1.ffill()
    linha3 = linha3.ffill()
    linha4 = linha4.ffill()
    
    novas_colunas = [
        f"{nome}_{ano}_{tipo}" if pd.notna(ano) else nome
        for nome, ano, tipo in zip(linha1, linha3, linha4)
    ]
    
    df.columns = novas_colunas
    
    df = df.iloc[5:].reset_index(drop=True)
    
    df.rename(columns={df.columns[0]: 'mesorregiao'}, inplace=True)
    
    df = df.dropna(how='all')
    
    df_long = df.melt(
        id_vars=['mesorregiao'],
        var_name='variavel_ano_tipo',
        value_name='valor'
    )
    
    df_long[['variavel', 'ano', 'tipo']] = df_long['variavel_ano_tipo'].str.rsplit('_', n=2, expand=True)
    
    df_long = df_long.drop(columns=['variavel_ano_tipo'])
    df_long['valor'] = pd.to_numeric(df_long['valor'], errors='coerce')


    df_final = df_long.pivot_table(
        index=['mesorregiao', 'ano', 'tipo'],
        columns='variavel',
        values='valor'
    ).reset_index()
    
    return df_final

In [49]:
dfs_arabica = []

for arquivo in arquivos_arabica:
    
    try:
        df = trata_excel(arquivo)
        
        dfs_arabica.append(df)
        
    except Exception as e:
        print(f"Erro em {arquivo}: {e}")

dfs_canephora = []

for arquivo in arquivos_canephora:
    
    try:
        df = trata_excel(arquivo)
        
        dfs_canephora.append(df)
        
    except Exception as e:
        print(f"Erro em {arquivo}: {e}")

In [50]:
dfs_arabica[0]

variavel,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare)
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,1000.0
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,393.0
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,295.0
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,272.0
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,383.0
...,...,...,...,...
819,Zona da Mata (MG),2020,Café (em grão) Arábica,1786.0
820,Zona da Mata (MG),2021,Café (em grão) Arábica,1087.0
821,Zona da Mata (MG),2022,Café (em grão) Arábica,1338.0
822,Zona da Mata (MG),2023,Café (em grão) Arábica,1299.0


In [51]:
dfs_canephora[0]

variavel,mesorregiao,ano,tipo,Variável - Área destinada à colheita (Hectares)
0,Baixo Amazonas (PA),2012,Café (em grão) Canephora,255.0
1,Baixo Amazonas (PA),2013,Café (em grão) Canephora,213.0
2,Baixo Amazonas (PA),2014,Café (em grão) Canephora,138.0
3,Baixo Amazonas (PA),2015,Café (em grão) Canephora,56.0
4,Baixo Amazonas (PA),2016,Café (em grão) Canephora,60.0
...,...,...,...,...
402,Zona da Mata (MG),2020,Café (em grão) Canephora,522.0
403,Zona da Mata (MG),2021,Café (em grão) Canephora,504.0
404,Zona da Mata (MG),2022,Café (em grão) Canephora,488.0
405,Zona da Mata (MG),2023,Café (em grão) Canephora,498.0


In [52]:
# Converter pandas DataFrames para Spark e combinar via join
sdf_arabica = [spark.createDataFrame(df.reset_index(drop=True)) for df in dfs_arabica]
sdf_canephora = [spark.createDataFrame(df.reset_index(drop=True)) for df in dfs_canephora]

df_final_arabica = reduce(lambda left, right: left.join(right, on=['mesorregiao', 'ano', 'tipo']), sdf_arabica)
df_final_canephora = reduce(lambda left, right: left.join(right, on=['mesorregiao', 'ano', 'tipo']), sdf_canephora)

In [53]:
df_final = df_final_arabica.unionByName(df_final_canephora)

In [54]:
df_final = df_final \
    .withColumn('Variável - Valor da produção (Mil Reais)',
                df_final['Variável - Valor da produção (Mil Reais)'] * 1000) \
    .withColumn('Variável - Quantidade produzida (Toneladas)',
                df_final['Variável - Quantidade produzida (Toneladas)'] * 1000)

In [55]:
df_final = df_final \
    .withColumnRenamed('Variável - Valor da produção (Mil Reais)',
                       'Variável - Valor da produção (Reais)') \
    .withColumnRenamed('Variável - Quantidade produzida (Toneladas)',
                       'Variável - Quantidade produzida (Quilogramas)')

In [56]:
df_final.toPandas()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas)
0,Central Espírito-santense (ES),2014,Café (em grão) Arábica,1277.0,330407000.0,53871.0,53871.0,68809000.0
1,Bauru (SP),2015,Café (em grão) Arábica,1257.0,116757000.0,16396.0,16386.0,20601000.0
2,Bauru (SP),2013,Café (em grão) Arábica,1311.0,106041000.0,18461.0,18115.0,23744000.0
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,272.0,1618000.0,2549.0,2549.0,693000.0
4,Araçatuba (SP),2020,Café (em grão) Arábica,1228.0,6510000.0,833.0,833.0,1023000.0
...,...,...,...,...,...,...,...,...
1224,Vale do Mucuri (MG),2013,Café (em grão) Canephora,611.0,41000.0,18.0,18.0,11000.0
1225,Vale do Mucuri (MG),2015,Café (em grão) Canephora,778.0,44000.0,18.0,18.0,14000.0
1226,Zona da Mata (MG),2024,Café (em grão) Canephora,2170.0,21537000.0,612.0,612.0,1328000.0
1227,Vale do Rio Doce (MG),2021,Café (em grão) Canephora,1862.0,113746000.0,7626.0,7626.0,14198000.0


In [57]:
caminho_shapefile = "dados-brutos/centroides/BR_Mesorregioes_2022/BR_Mesorregioes_2022.shp"
mesorregioes = gpd.read_file(caminho_shapefile)
mesorregioes['centroide'] = mesorregioes.geometry.centroid
mesorregioes['latitude'] = mesorregioes.centroide.y
mesorregioes['longitude'] = mesorregioes.centroide.x

mesorregioes['NM_MESO'] = mesorregioes['NM_MESO'] + ' (' + mesorregioes['SIGLA_UF'] + ')'

mesorregioes = mesorregioes[['NM_MESO', 'latitude', 'longitude']]

/var/folders/wp/hrfyx8x11jlbkyy3vr606yph0000gn/T/ipykernel_17705/726320708.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  mesorregioes['centroide'] = mesorregioes.geometry.centroid


In [58]:
mesorregioes.rename(columns={'NM_MESO': 'mesorregiao'}, inplace=True)
mesorregioes_spark = spark.createDataFrame(mesorregioes)
mesorregioes.head()

,mesorregiao,latitude,longitude
0,Madeira-Guaporé (RO),-10.302659,-64.055361
1,Leste Rondoniense (RO),-11.406309,-61.861723
2,Vale do Juruá (AC),-8.539777,-71.773867
3,Vale do Acre (AC),-9.941532,-69.066201
4,Norte Amazonense (AM),-0.615226,-65.567522


In [59]:
df_final = df_final.join(mesorregioes_spark, on='mesorregiao', how='left')

In [60]:
df_final.toPandas().head()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude
0,Itapetininga (SP),2013,Café (em grão) Arábica,906.0,5320000.0,1526.0,1526.0,1382000.0,-23.88362,-48.641474
1,Itapetininga (SP),2015,Café (em grão) Arábica,1400.0,11773000.0,1537.0,1537.0,2152000.0,-23.88362,-48.641474
2,Itapetininga (SP),2018,Café (em grão) Arábica,1409.0,11579000.0,1205.0,1205.0,1698000.0,-23.88362,-48.641474
3,Itapetininga (SP),2022,Café (em grão) Arábica,1868.0,47672000.0,1288.0,1288.0,2406000.0,-23.88362,-48.641474
4,Itapetininga (SP),2023,Café (em grão) Arábica,1510.0,25508000.0,1246.0,1246.0,1882000.0,-23.88362,-48.641474


In [61]:
def trata_temperatura(caminho):

    cabecalho = pd.read_csv(
        caminho,
        sep=';',
        encoding='latin1',
        nrows=8,
        header=None
    )


    latitude = cabecalho.iloc[4, 1]
    longitude = cabecalho.iloc[5, 1]
    altitude  = cabecalho.iloc[6, 1]


    df = pd.read_csv(
    caminho,
    sep=';',
    encoding='latin1',
    skiprows=8  
    )

    df['latitude'] = latitude
    df['longitude'] = longitude
    df['altitude'] = altitude

    nome_arquivo = os.path.basename(caminho)

    partes = nome_arquivo.split('_')

    regiao = partes[1]
    uf = partes[2]
    cidade = partes[4]
    ano = partes[5]

    df['regiao'] = regiao
    df['uf'] = uf
    df['cidade'] = cidade
    df['ano'] = ano

    if ano[-4:] in ['2019', '2020', '2021', '2022', '2023', '2024']:
        df['DATA (YYYY-MM-DD)'] = pd.to_datetime(df['Data']).dt.date

    df = df[['DATA (YYYY-MM-DD)', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)', 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
          'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)', 'UMIDADE RELATIVA DO AR, HORARIA (%)', 
          'regiao', 'uf', 'cidade', 'ano', 'latitude', 'longitude', 'altitude']]
    
    cols = [
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
        'altitude'
    ]

    for col in cols:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.'),
            errors='coerce'
        )

    df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].astype(float)
    df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'] = df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'].astype(float)
    df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'] = df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'].astype(float)
    df['altitude'] = df['altitude'].astype(float)

    df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].clip(lower=0)
    df = df[df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'] >= -100]
    df = df[df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'] >= -100]
    df = df[df['UMIDADE RELATIVA DO AR, HORARIA (%)'] >= 0]
    df = df[df['UMIDADE RELATIVA DO AR, HORARIA (%)'] <= 100]
    df = df[df['altitude'] >= 0]

    df_diario = df.groupby('DATA (YYYY-MM-DD)').agg({
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'sum',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'max',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'min',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'mean',
        'altitude': 'mean'
    }).reset_index()


    df_diario['DATA (YYYY-MM-DD)'] = pd.to_datetime(df_diario['DATA (YYYY-MM-DD)'])
    mask = df_diario['DATA (YYYY-MM-DD)'].dt.month >= 10
    soma = df_diario.loc[mask, 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].sum()
    df_diario['precipitacao_floracao'] = soma

    df_final = df_diario.agg({
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'sum',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'mean',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'mean',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'mean',
        'precipitacao_floracao': 'mean',
        'altitude': 'mean'
    }).to_frame().T


    df_final['regiao'] = df['regiao'].iloc[0]
    df_final['uf'] = df['uf'].iloc[0]
    df_final['cidade'] = df['cidade'].iloc[0]
    df_final['ano'] = df['ano'].iloc[0]
    df_final['latitude'] = df['latitude'].iloc[0]
    df_final['longitude'] = df['longitude'].iloc[0]

    df_final = df_final.rename(columns={
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'precipitacao_total (mm)',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'temperatura_maxima (°C)',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'temperatura_minima (°C)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade_relativa (%)'})

    df_final['ano'] = pd.to_datetime(df_final['ano'], format='%d-%m-%Y')
    df_final['ano'] = df_final['ano'].dt.year

    return df_final

In [62]:
dfs = []

for root, dirs, files in os.walk('dados-brutos/dados-climaticos'):
    for file in files:
        if file.endswith('.CSV'):
            
            caminho = os.path.join(root, file)
            
            try:
                df = trata_temperatura(caminho)
                
                if df is not None:
                    dfs.append(df)
            
            except Exception as e:
                print(f"Erro em {caminho}: {e}")

df_temp_pd = pd.concat(dfs, ignore_index=True)

# Converter latitude e longitude para numérico antes de criar o Spark DataFrame
for col in ['latitude', 'longitude']:
    df_temp_pd[col] = pd.to_numeric(
        df_temp_pd[col].astype(str).str.replace(',', '.'),
        errors='coerce'
    ).astype(float)

df_temp = spark.createDataFrame(df_temp_pd)

Erro em dados-brutos/dados-climaticos/2013/INMET_SE_MG_F501_BELO HORIZONTE - CERCADINHO_27-12-2013_A_31-12-2013.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2013/INMET_NE_RN_A302_ARQ.SAO PEDRO E SAO PAULO_01-01-2013_A_31-12-2013.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2013/INMET_N_PA_A234_TUCUMA_01-01-2013_A_31-12-2013.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2014/2014/INMET_N_PA_A234_TUCUMA_01-01-2014_A_31-12-2014.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2014/2014/INMET_NE_RN_A302_ARQ.SAO PEDRO E SAO PAULO_01-01-2014_A_31-12-2014.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2014/2014/INMET_S_RS_A804_SANTANA DO LIVRAMENTO_01-01-2014_A_31-12-2014.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2022/INMET_N_TO_A049_COLINAS DO TOCANTINS_

In [63]:
df_temp.toPandas().tail()

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude
6745,569.4,30.264793,18.981361,67.604282,206.2,387.00,CO,MS,BATAGUASSU,2016,-21.751389,-52.470556
6746,1140.8,28.080328,17.037978,62.260310,507.2,1159.54,CO,DF,BRASILIA,2016,-15.789444,-47.925833
6747,1618.8,26.074238,12.508310,77.403441,584.0,1150.00,SE,MG,CALDAS,2016,-21.918056,-46.383056
6748,1895.6,30.111202,17.782240,79.676700,620.0,591.00,CO,MT,COMODORO,2016,-13.708611,-59.762778
6749,617.0,32.302265,19.851133,69.279628,18.6,518.00,CO,MT,NOVA UBIRATA,2016,-13.686389,-54.956111


In [64]:
from math import radians, sin, cos, sqrt, atan2

@udf(returnType=DoubleType())
def haversine(lat1, lon1, lat2, lon2):
    if None in [lat1, lon1, lat2, lon2]:
        return None
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

In [65]:
mesorregioes

,mesorregiao,latitude,longitude
0,Madeira-Guaporé (RO),-10.302659,-64.055361
1,Leste Rondoniense (RO),-11.406309,-61.861723
2,Vale do Juruá (AC),-8.539777,-71.773867
3,Vale do Acre (AC),-9.941532,-69.066201
4,Norte Amazonense (AM),-0.615226,-65.567522
...,...,...,...
134,Norte Goiano (GO),-13.895544,-48.347095
135,Centro Goiano (GO),-15.981377,-49.778494
136,Leste Goiano (GO),-15.279044,-47.512262
137,Sul Goiano (GO),-17.743173,-50.504003


In [66]:
# Conversão de latitude e longitude já realizada antes da criação do Spark DataFrame (célula anterior)

In [67]:
df_temp_2019 = df_temp.filter(F.col('ano') == 2019)
df_temp_teste = df_temp_2019

In [68]:
df_temp_teste.toPandas()

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude
0,337.4,32.609497,22.196648,73.092404,334.4,147.75,CO,MT,PORTO ESTRELA,2019,-15.324722,-57.225833
1,1038.2,32.326849,20.144658,69.628680,391.6,374.00,CO,MS,SELVIRIA,2019,-20.351444,-51.430222
2,69.6,33.727072,20.518508,67.256851,1.6,515.30,NE,PI,CARACOL,2019,-9.285833,-43.324444
3,1995.8,27.203836,16.233973,72.005500,580.0,245.50,S,RS,SAO LUIZ GONZAGA,2019,-28.417113,-54.962403
4,479.8,32.939726,20.623562,73.671252,29.2,182.00,NE,BA,RIBEIRA DO AMPARO,2019,-11.058611,-38.444167
...,...,...,...,...,...,...,...,...,...,...,...,...
583,1323.2,25.783014,15.574247,75.913470,395.0,106.99,S,RS,RIO PARDO,2019,-29.872113,-52.381980
584,1036.6,31.443288,18.047945,64.427910,499.0,640.85,SE,MG,UNAI,2019,-16.554101,-46.881935
585,1107.2,31.431621,24.712253,65.795276,19.6,3.72,NE,SE,ARACAJU,2019,-10.952413,-37.054330
586,1252.8,28.580548,19.535616,79.014798,61.4,756.00,NE,CE,TIANGUA,2019,-3.732169,-41.011881


In [69]:
# Encontrar a mesorregião mais próxima de cada estação via cross join + window
estacoes = df_temp_2019.select('regiao', 'uf', 'cidade', 'latitude', 'longitude').distinct()

cross = estacoes.crossJoin(
    mesorregioes_spark.select(
        'mesorregiao',
        F.col('latitude').alias('lat_meso'),
        F.col('longitude').alias('lon_meso')
    )
)

cross = cross.withColumn('distancia', haversine('latitude', 'longitude', 'lat_meso', 'lon_meso'))

window_estacao = Window.partitionBy('regiao', 'uf', 'cidade')
cross = cross.withColumn('min_dist', F.min('distancia').over(window_estacao))

meso_por_estacao = cross.filter(F.col('distancia') == F.col('min_dist')) \
                        .select('regiao', 'uf', 'cidade', 'mesorregiao')

# Adicionar mesorregiao ao df_temp_teste para exibição consistente com o original
df_temp_teste = df_temp_2019.join(meso_por_estacao, on=['regiao', 'uf', 'cidade'], how='left')

print(mesorregioes)

                mesorregiao   latitude  longitude
0      Madeira-Guaporé (RO) -10.302659 -64.055361
1    Leste Rondoniense (RO) -11.406309 -61.861723
2        Vale do Juruá (AC)  -8.539777 -71.773867
3         Vale do Acre (AC)  -9.941532 -69.066201
4     Norte Amazonense (AM)  -0.615226 -65.567522
..                      ...        ...        ...
134       Norte Goiano (GO) -13.895544 -48.347095
135      Centro Goiano (GO) -15.981377 -49.778494
136       Leste Goiano (GO) -15.279044 -47.512262
137         Sul Goiano (GO) -17.743173 -50.504003
138   Distrito Federal (DF) -15.781166 -47.796851

[139 rows x 3 columns]


In [70]:
df_temp_teste.toPandas()

,regiao,uf,cidade,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,ano,latitude,longitude,mesorregiao
0,SE,RJ,MACAE,1537.0,29.414795,20.224658,80.667009,551.0,28.00,2019,-22.376111,-41.811944,Baixadas (RJ)
1,SE,SP,SAO SIMAO,664.0,30.803846,17.313736,63.225083,565.8,620.00,2019,-21.461111,-47.579444,Ribeirão Preto (SP)
2,NE,SE,ITABAIANINHA,264.2,31.446233,22.586644,75.463216,34.4,204.80,2019,-11.272500,-37.795000,Agreste Sergipano (SE)
3,N,PA,SANTANA DO ARAGUAIA,1661.0,34.571233,21.508493,69.932900,655.6,176.75,2019,-9.338611,-50.350280,Ocidental do Tocantins (TO)
4,CO,MT,SALTO DO CEU,142.8,32.765205,20.208767,73.995434,128.0,300.83,2019,-15.124722,-58.127222,Sudoeste Mato-grossense (MT)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
583,S,SC,DIONISIO CERQUEIRA,2038.4,24.954795,16.107397,72.263282,565.6,807.54,2019,-26.286562,-53.633114,Sudoeste Paranaense (PR)
584,SE,MG,GUANHAES,865.8,28.693151,16.971781,70.193817,380.4,852.68,2019,-18.786842,-42.942921,Vale do Rio Doce (MG)
585,SE,SP,BEBDOURO,1345.4,31.146575,19.126301,65.149484,496.8,790.00,2019,-20.949167,-48.490000,Ribeirão Preto (SP)
586,CO,GO,CRISTALINA,1193.6,27.721644,17.381918,62.616536,559.6,1211.08,2019,-16.784896,-47.612966,Distrito Federal (DF)


In [71]:
df_temp = df_temp.join(meso_por_estacao, on=['regiao', 'uf', 'cidade'], how='left')

In [72]:
df_temp.toPandas()

,regiao,uf,cidade,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,ano,latitude,longitude,mesorregiao
0,SE,RJ,VILA MILITAR,1130.0,29.374247,19.136986,71.020260,487.8,45.0,2013,-22.860833,-43.411111,None
1,SE,RJ,VILA MILITAR,923.2,30.881644,19.123288,66.378164,210.0,45.0,2014,-22.860833,-43.411111,None
2,SE,RJ,MACAE,1341.8,29.092953,20.262752,79.386859,645.2,32.0,2013,-22.383333,-41.816667,Baixadas (RJ)
3,SE,RJ,MACAE,605.0,29.709041,19.858630,76.307808,157.6,32.0,2014,-22.383333,-41.816667,Baixadas (RJ)
4,S,RS,SANTANA DO LIVRAMENTO,792.4,22.774383,13.088580,70.618495,9.4,196.0,2013,-30.842500,-55.612778,Sudoeste Rio-grandense (RS)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6745,CO,GO,CRISTALINA,1567.6,27.633060,17.398634,61.980760,686.2,1202.0,2016,-16.788056,-47.611667,Distrito Federal (DF)
6746,S,SC,DIONISIO CERQUEIRA,1334.8,23.609016,14.962295,72.319870,556.0,810.0,2016,-26.286389,-53.632778,Sudoeste Paranaense (PR)
6747,SE,MG,GUANHAES,1448.2,27.760929,16.575137,74.656781,530.0,860.0,2016,-18.786667,-42.943056,Vale do Rio Doce (MG)
6748,SE,SP,BEBDOURO,153.6,30.548980,20.653061,70.901282,153.6,590.0,2016,-20.949167,-48.489722,Ribeirão Preto (SP)


In [73]:
df_temp.select('mesorregiao').distinct().count()

140

In [74]:
mesorregioes['mesorregiao'].nunique()

139

In [75]:
df_temp.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6750 entries, 0 to 6749
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   regiao                   6750 non-null   object 
 1   uf                       6750 non-null   object 
 2   cidade                   6750 non-null   object 
 3   precipitacao_total (mm)  6750 non-null   float64
 4   temperatura_maxima (°C)  6750 non-null   float64
 5   temperatura_minima (°C)  6750 non-null   float64
 6   umidade_relativa (%)     6750 non-null   float64
 7   precipitacao_floracao    6750 non-null   float64
 8   altitude                 6750 non-null   float64
 9   ano                      6750 non-null   int64  
 10  latitude                 6750 non-null   float64
 11  longitude                6750 non-null   float64
 12  mesorregiao              6607 non-null   object 
dtypes: float64(8), int64(1), object(4)
memory usage: 685.7+ KB


In [76]:
df_temp_agg = df_temp.filter(F.col('mesorregiao').isNotNull()).groupBy('mesorregiao', 'ano').agg(
    F.mean('precipitacao_total (mm)').alias('precipitacao_total (mm)'),
    F.mean('temperatura_maxima (°C)').alias('temperatura_maxima (°C)'),
    F.mean('temperatura_minima (°C)').alias('temperatura_minima (°C)'),
    F.mean('umidade_relativa (%)').alias('umidade_relativa (%)'),
    F.mean('precipitacao_floracao').alias('precipitacao_floracao'),
    F.mean('altitude').alias('altitude')
)

In [77]:
df_temp_agg.toPandas()

,mesorregiao,ano,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Sul de Roraima (RR),2023,2436.400000,32.141644,23.735616,81.686311,206.600000,29.880000
1,Ocidental do Tocantins (TO),2016,944.355556,33.940612,21.996183,71.659190,414.177778,197.666667
2,São Francisco Pernambucano (PE),2023,188.000000,33.796369,22.508460,58.700215,44.800000,361.820000
3,Central Potiguar (RN),2020,314.133333,34.558318,23.576429,58.720447,2.200000,69.210000
4,Vale do Mucuri (MG),2016,675.300000,29.526862,19.853989,70.952704,361.200000,341.500000
...,...,...,...,...,...,...,...,...
1769,Sul do Amapá (AP),2014,2422.400000,31.366301,23.804384,78.389840,114.000000,27.000000
1770,Litoral Norte Espírito-santense (ES),2020,556.400000,28.809976,21.838121,62.473098,93.750000,23.910000
1771,Metropolitana de Fortaleza (CE),2014,379.600000,31.360882,24.395588,69.865402,76.800000,26.450000
1772,Noroeste Fluminense (RJ),2024,488.800000,31.883240,22.292737,68.024103,290.000000,46.000000


In [78]:
df_temp_agg.filter(F.col('mesorregiao').contains('(SC)')).toPandas()

,mesorregiao,ano,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Vale do Itajaí (SC),2012,1788.20,25.363183,15.920833,82.360142,467.25,295.032500
1,Serrana (SC),2024,1287.40,23.483354,13.459652,77.357530,254.45,959.515000
2,Serrana (SC),2023,2045.80,23.143518,12.929490,79.914410,814.50,959.515000
3,Oeste Catarinense (SC),2016,1435.75,23.608639,13.549296,78.409492,404.55,852.500000
4,Grande Florianópolis (SC),2021,1593.80,22.706712,14.867123,83.035426,349.30,442.935000
...,...,...,...,...,...,...,...,...
73,Norte Catarinense (SC),2013,1453.40,22.961233,11.929863,85.089802,335.30,838.500000
74,Norte Catarinense (SC),2022,722.70,23.559410,13.150161,84.788793,324.00,799.790000
75,Norte Catarinense (SC),2020,573.30,24.492896,12.157787,83.252283,160.60,828.235000
76,Serrana (SC),2014,1052.80,23.535920,13.466468,78.397456,395.00,922.333333


In [79]:
df_final.toPandas().head()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude
0,Itapetininga (SP),2013,Café (em grão) Arábica,906.0,5320000.0,1526.0,1526.0,1382000.0,-23.88362,-48.641474
1,Itapetininga (SP),2015,Café (em grão) Arábica,1400.0,11773000.0,1537.0,1537.0,2152000.0,-23.88362,-48.641474
2,Itapetininga (SP),2018,Café (em grão) Arábica,1409.0,11579000.0,1205.0,1205.0,1698000.0,-23.88362,-48.641474
3,Itapetininga (SP),2022,Café (em grão) Arábica,1868.0,47672000.0,1288.0,1288.0,2406000.0,-23.88362,-48.641474
4,Itapetininga (SP),2023,Café (em grão) Arábica,1510.0,25508000.0,1246.0,1246.0,1882000.0,-23.88362,-48.641474


In [80]:
df_temp_agg.toPandas().head()

,mesorregiao,ano,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Sul de Roraima (RR),2023,2436.400000,32.141644,23.735616,81.686311,206.600000,29.880000
1,Ocidental do Tocantins (TO),2016,944.355556,33.940612,21.996183,71.659190,414.177778,197.666667
2,São Francisco Pernambucano (PE),2023,188.000000,33.796369,22.508460,58.700215,44.800000,361.820000
3,Central Potiguar (RN),2020,314.133333,34.558318,23.576429,58.720447,2.200000,69.210000
4,Vale do Mucuri (MG),2016,675.300000,29.526862,19.853989,70.952704,361.200000,341.500000


In [81]:
df_final = df_final.withColumn('ano', F.col('ano').cast('int'))
df_final = df_final.join(df_temp_agg, on=['mesorregiao', 'ano'], how='left')

In [82]:
df_final.toPandas()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Vale do Mucuri (MG),2016,Café (em grão) Arábica,1033.0,3.774700e+07,5300.0,5300.0,5473000.0,-17.633398,-41.194362,675.300000,29.526862,19.853989,70.952704,361.200000,341.500000
1,Araçatuba (SP),2024,Café (em grão) Arábica,870.0,6.689000e+06,640.0,640.0,557000.0,-21.066546,-50.813202,1058.200000,33.358470,19.729326,62.165858,546.733333,372.966667
2,Bauru (SP),2017,Café (em grão) Arábica,1360.0,1.310150e+08,15123.0,15123.0,20573000.0,-22.489939,-48.999826,1503.466667,28.202648,16.930685,71.271601,514.466667,661.666667
3,Norte Pioneiro Paranaense (PR),2014,Café (em grão) Arábica,1204.0,1.531580e+08,21802.0,21802.0,26245000.0,-23.475189,-50.300861,1213.133333,27.207489,16.669954,71.840952,262.800000,628.000000
4,Vale do Mucuri (MG),2017,Café (em grão) Arábica,719.0,9.679000e+06,2053.0,2053.0,1477000.0,-17.633398,-41.194362,574.800000,28.785668,19.282934,74.141271,204.600000,341.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1224,Norte Cearense (CE),2018,Café (em grão) Canephora,533.0,4.600000e+04,15.0,15.0,8000.0,-4.037035,-39.131272,1053.800000,29.731057,21.305217,78.075153,57.000000,70.100000
1225,Sul Amazonense (AM),2017,Café (em grão) Canephora,1432.0,1.089000e+06,304.0,294.0,421000.0,-7.059497,-63.380741,2371.950000,32.496775,22.567756,80.412304,590.000000,91.750000
1226,Litoral Norte Espírito-santense (ES),2020,Café (em grão) Canephora,2290.0,1.174119e+09,88903.0,88903.0,203551000.0,-18.951717,-40.122243,556.400000,28.809976,21.838121,62.473098,93.750000,23.910000
1227,Noroeste Fluminense (RJ),2024,Café (em grão) Canephora,2500.0,1.157000e+06,26.0,26.0,65000.0,-21.312889,-41.945746,488.800000,31.883240,22.292737,68.024103,290.000000,46.000000


In [83]:
df_final.filter(F.col('altitude').isNull() | F.isnan(F.col('altitude'))).toPandas()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Sudoeste Amazonense (AM),2022,Café (em grão) Arábica,2000.0,6.700000e+04,4.0,4.0,8000.0,-5.180785,-69.356680,NaN,NaN,NaN,NaN,NaN,NaN
1,Assis (SP),2021,Café (em grão) Arábica,1608.0,2.917180e+08,17432.0,17415.0,28009000.0,-22.768414,-50.121347,NaN,NaN,NaN,NaN,NaN,NaN
2,Norte Fluminense (RJ),2016,Café (em grão) Arábica,1540.0,4.090000e+05,63.0,63.0,97000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
3,Assis (SP),2022,Café (em grão) Arábica,1579.0,3.054490e+08,17437.0,17420.0,27514000.0,-22.768414,-50.121347,NaN,NaN,NaN,NaN,NaN,NaN
4,Sudoeste Amazonense (AM),2021,Café (em grão) Arábica,1300.0,1.280000e+05,21.0,10.0,13000.0,-5.180785,-69.356680,NaN,NaN,NaN,NaN,NaN,NaN
5,Campinas (SP),2021,Café (em grão) Arábica,1589.0,1.254622e+09,53648.0,53532.0,85049000.0,-22.220282,-46.989825,NaN,NaN,NaN,NaN,NaN,NaN
6,Norte Fluminense (RJ),2015,Café (em grão) Arábica,1540.0,2.740000e+05,63.0,63.0,97000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
7,Norte Fluminense (RJ),2017,Café (em grão) Arábica,1585.0,7.200000e+05,65.0,65.0,103000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
8,Norte Fluminense (RJ),2018,Café (em grão) Arábica,1492.0,6.060000e+05,63.0,63.0,94000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
9,Norte Fluminense (RJ),2014,Café (em grão) Arábica,1540.0,2.740000e+05,93.0,63.0,97000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
df_final = df_final.orderBy('mesorregiao', 'ano', 'tipo')

colunas_climate = [
    'precipitacao_total (mm)',
    'temperatura_maxima (°C)',
    'temperatura_minima (°C)',
    'umidade_relativa (%)',
    'altitude'
]

# Converter NaN para null antes do bfill (necessário para ignorenulls funcionar corretamente)
for col_name in colunas_climate:
    df_final = df_final.withColumn(
        col_name,
        F.when(F.isnan(df_final[col_name]) | df_final[col_name].isNull(), None)
         .otherwise(df_final[col_name])
    )

window_bfill = Window.partitionBy('mesorregiao').orderBy('ano', 'tipo') \
                     .rowsBetween(0, Window.unboundedFollowing)

for col_name in colunas_climate:
    df_final = df_final.withColumn(
        col_name,
        F.first(df_final[col_name], ignorenulls=True).over(window_bfill)
    )

In [85]:
df_final.filter(F.col('altitude').isNull()).toPandas()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Centro-Sul Paranaense (PR),2012,Café (em grão) Arábica,1000.0,5000.0,1.0,1.0,1000.0,-25.487813,-51.972895,NaN,NaN,NaN,NaN,NaN,NaN
1,Sudoeste Amazonense (AM),2021,Café (em grão) Arábica,1300.0,128000.0,21.0,10.0,13000.0,-5.180785,-69.356680,NaN,NaN,NaN,NaN,NaN,NaN
2,Sudoeste Amazonense (AM),2022,Café (em grão) Arábica,2000.0,67000.0,4.0,4.0,8000.0,-5.180785,-69.356680,NaN,NaN,NaN,NaN,NaN,NaN
3,Sudoeste Amazonense (AM),2022,Café (em grão) Canephora,1250.0,79000.0,19.0,12.0,15000.0,-5.180785,-69.356680,NaN,NaN,NaN,NaN,NaN,NaN
4,Sudoeste Amazonense (AM),2023,Café (em grão) Canephora,1176.0,156000.0,18.0,17.0,20000.0,-5.180785,-69.356680,NaN,NaN,NaN,NaN,NaN,NaN
5,Sudoeste Amazonense (AM),2024,Café (em grão) Canephora,1167.0,194000.0,19.0,18.0,21000.0,-5.180785,-69.356680,NaN,NaN,NaN,NaN,NaN,NaN


In [86]:
df_final.filter(F.col('mesorregiao').contains('Sudoeste Amazonense')).toPandas()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Sudoeste Amazonense (AM),2012,Café (em grão) Canephora,1000.0,241000.0,131.0,108.0,108000.0,-5.180785,-69.35668,693.2,32.257724,22.260976,81.921297,608.8,143.00
1,Sudoeste Amazonense (AM),2013,Café (em grão) Canephora,800.0,168000.0,108.0,70.0,56000.0,-5.180785,-69.35668,2369.0,31.049315,22.024384,80.183440,814.0,143.00
2,Sudoeste Amazonense (AM),2014,Café (em grão) Canephora,1200.0,266000.0,80.0,80.0,96000.0,-5.180785,-69.35668,2218.6,30.990909,22.102204,70.406612,275.2,143.00
3,Sudoeste Amazonense (AM),2015,Café (em grão) Canephora,1200.0,137000.0,30.0,30.0,36000.0,-5.180785,-69.35668,2779.4,31.900548,22.301644,65.677770,808.0,143.00
4,Sudoeste Amazonense (AM),2016,Café (em grão) Canephora,1200.0,137000.0,30.0,30.0,36000.0,-5.180785,-69.35668,786.4,31.425743,23.499010,71.101247,0.0,143.00
5,Sudoeste Amazonense (AM),2017,Café (em grão) Canephora,1200.0,162000.0,30.0,30.0,36000.0,-5.180785,-69.35668,96.4,31.576667,23.525000,72.051523,NaN,121.54
6,Sudoeste Amazonense (AM),2020,Café (em grão) Canephora,1194.0,144000.0,31.0,31.0,37000.0,-5.180785,-69.35668,96.4,31.576667,23.525000,72.051523,0.0,121.54
7,Sudoeste Amazonense (AM),2021,Café (em grão) Arábica,1300.0,128000.0,21.0,10.0,13000.0,-5.180785,-69.35668,NaN,NaN,NaN,NaN,NaN,NaN
8,Sudoeste Amazonense (AM),2022,Café (em grão) Arábica,2000.0,67000.0,4.0,4.0,8000.0,-5.180785,-69.35668,NaN,NaN,NaN,NaN,NaN,NaN
9,Sudoeste Amazonense (AM),2022,Café (em grão) Canephora,1250.0,79000.0,19.0,12.0,15000.0,-5.180785,-69.35668,NaN,NaN,NaN,NaN,NaN,NaN


In [87]:
df_final = df_final \
    .withColumn('aproveitamento_colheita',
                df_final['Variável - Área colhida (Hectares)'] /
                df_final['Variável - Área destinada à colheita (Hectares)']) \
    .withColumn('amplitude_termica',
                df_final['temperatura_maxima (°C)'] - df_final['temperatura_minima (°C)']) \
    .withColumn('preco_medio_kg',
                df_final['Variável - Valor da produção (Reais)'] /
                df_final['Variável - Quantidade produzida (Quilogramas)'])

In [88]:
df_final.toPandas()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,aproveitamento_colheita,amplitude_termica,preco_medio_kg
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,1000.0,3.600000e+04,2.0,2.0,2000.0,-7.029713,-35.819545,768.800000,28.629075,20.254114,82.511918,54.100000,559.810000,1.0,8.374962,18.000000
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,393.0,3.851000e+06,2733.0,2733.0,1075000.0,-8.510505,-36.416511,232.066667,29.143534,18.353461,70.905596,12.333333,684.486667,1.0,10.790073,3.582326
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,295.0,2.439000e+06,2615.0,2615.0,771000.0,-8.510505,-36.416511,618.066667,29.036885,18.879336,73.331692,106.800000,684.486667,1.0,10.157548,3.163424
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,272.0,1.618000e+06,2549.0,2549.0,693000.0,-8.510505,-36.416511,606.266667,28.628146,18.599447,74.939452,116.933333,684.486667,1.0,10.028699,2.334776
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,383.0,2.325000e+06,2265.0,2265.0,867000.0,-8.510505,-36.416511,403.000000,29.760529,18.845805,70.811290,57.933333,684.486667,1.0,10.914724,2.681661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1224,Zona da Mata (MG),2022,Café (em grão) Canephora,2746.0,1.532700e+07,488.0,488.0,1340000.0,-21.010901,-42.819740,1326.733333,29.056661,17.429160,76.393595,734.600000,463.856667,1.0,11.627501,11.438060
1225,Zona da Mata (MG),2023,Café (em grão) Arábica,1299.0,3.658570e+09,201832.0,201832.0,262264000.0,-21.010901,-42.819740,1327.266667,29.921461,18.544932,77.370603,405.400000,463.856667,1.0,11.376530,13.949951
1226,Zona da Mata (MG),2023,Café (em grão) Canephora,2371.0,1.064300e+07,498.0,498.0,1181000.0,-21.010901,-42.819740,1327.266667,29.921461,18.544932,77.370603,405.400000,463.856667,1.0,11.376530,9.011854
1227,Zona da Mata (MG),2024,Café (em grão) Arábica,1376.0,5.693510e+09,205375.0,205375.0,282595000.0,-21.010901,-42.819740,1511.800000,30.457941,18.854546,76.623899,654.066667,463.856667,1.0,11.603395,20.147243
